# DEGST — Kaggle T4×2 runner

Chạy từng **pha** (P0–P4, chọn bằng `PHASE`) của `experiments/plan.py` trên 2 GPU T4 (mỗi GPU một shard `i/2`). Runner **resumable**: run nào đã có `results/<run_id>.json` thì bỏ qua, nên có thể chạy lại nhiều session (giới hạn ~12h/session) bằng cách gắn output session trước làm Input dataset (biến `RESTORE_FROM`).

Settings: Accelerator = **GPU T4 x2**, Internet = **On**.

In [ ]:
import os, subprocess, sys, json, time, zipfile, shutil
from pathlib import Path

# ---- cấu hình ----
REPO_URL     = 'https://github.com/nWoWolfpac/DEGST_fixed.git'
BRANCH       = 'hung'
PHASE        = 'P0'             # xem bảng PHASES bên dưới; đổi giá trị này giữa các session
PHASES = {
    'P0': dict(suite='quick',       post=[],  note='smoke 4 run x 5 epoch, kiểm tra môi trường (~vài phút)'),
    'P1': dict(suite='study',       post=['aggregate.py'], note='92 run, ~34h/3090 -> dự kiến 2-3 session T4x2; chạy lại P1 tới khi status đủ'),
    'P2': dict(suite='regimes_tin', post=['aggregate.py'], note='Tiny ImageNet 10 run, ~14h/3090 -> 1-2 session'),
    'P3': dict(suite='screen',      post=['aggregate.py --phase2'], note='24 run phase 2; đọc screen_decision.json'),
    'P4': dict(suite='beta_sweep',  post=['aggregate.py --phase2'], note='CHỈ chạy nếu stop rule của P3 cho phép'),
}
TIME_BUDGET_H = 11.0            # dừng trước khi Kaggle ngắt session (12h)
NUM_WORKERS  = 2                # Kaggle chỉ có 4 CPU: 2 worker/GPU
RESTORE_FROM = None             # vd '/kaggle/input/degst-results-v1' (output session trước, chứa thư mục results/)
USE_TOKEN    = True             # repo private: cần Kaggle secret 'github_token'

SUITE = PHASES[PHASE]['suite']
print(PHASE, '->', SUITE, '|', PHASES[PHASE]['note'])
ROOT = Path('/kaggle/working/DEGST_fixed')

In [ ]:
# Repo private: cần Kaggle secret 'github_token' (Add-ons > Secrets). Token đi qua GIT_ASKPASS, không nằm trong URL.
askpass = Path('/kaggle/working/.github_git_askpass.py')
env = {**os.environ, 'GIT_TERMINAL_PROMPT': '0'}
token = ''
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('github_token').strip()
except Exception as e:
    print('Không đọc được secret github_token:', e)
if token:   # chẩn đoán: không in token, chỉ in dạng và phản hồi của GitHub
    import urllib.request, urllib.error
    print(f'token: len={len(token)} prefix={token[:4]!r} có khoảng trắng/xuống dòng bên trong={any(c.isspace() for c in token)}')
    try:
        r = urllib.request.urlopen(urllib.request.Request('https://api.github.com/user', headers={'Authorization': f'Bearer {token}'}))
        print('GitHub API: 200, login =', json.load(r)['login'])
    except urllib.error.HTTPError as e:
        print('GitHub API:', e.code, e.reason, '-> token bị từ chối (hết hạn/thu hồi/sai)')
assert token or not USE_TOKEN, "Thiếu Kaggle secret 'github_token' (hoặc đặt USE_TOKEN=False nếu repo public)"
if token and USE_TOKEN:
    askpass.write_text("#!/usr/bin/env python3\nimport os,sys\nprint('x-access-token' if 'Username' in sys.argv[1] else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
    askpass.chmod(0o700)
    env.update(GIT_ASKPASS=str(askpass), GITHUB_TOKEN_RUNTIME=token)
try:
    if (ROOT/'.git').is_dir():
        subprocess.run(['git','-C',str(ROOT),'pull','--ff-only','origin',BRANCH], env=env, check=True)
    else:
        subprocess.run(['git','clone','--branch',BRANCH,REPO_URL,str(ROOT)], env=env, check=True)
finally:
    askpass.unlink(missing_ok=True); token = None
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
print('Commit:', subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True).stdout.strip())

## Cài đặt thư viện
Kaggle đã có torch/torchvision/numpy/matplotlib/scipy; chỉ thiếu `torch-pruning`.

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','torch-pruning>=1.5','scipy'], check=True)
import torch
print('torch', torch.__version__, '| GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() >= 1, 'Bật GPU trong Settings > Accelerator'
N_GPU = torch.cuda.device_count()

## Khôi phục kết quả cũ (nếu có) + unit test

In [ ]:
if RESTORE_FROM:
    src = Path(RESTORE_FROM)
    cand = [p for p in [src/'results', *src.rglob('results')] if p.is_dir()]
    assert cand, f'Không thấy thư mục results trong {src}'
    shutil.copytree(cand[0], ROOT/'results', dirs_exist_ok=True)
    print('restored', len(list((ROOT/'results').glob('*.json'))), 'json từ', cand[0])
r = subprocess.run([sys.executable,'tests/test_degst.py'], capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:])
assert r.returncode == 0, 'unit test fail'

In [ ]:
print(subprocess.run([sys.executable,'-m','experiments.plan','status','all'],capture_output=True,text=True).stdout)
print(subprocess.run([sys.executable,'-m','experiments.plan','status',SUITE],capture_output=True,text=True).stdout)

## Chạy suite trên 2 GPU
Mỗi GPU chạy một shard (`--shard i/N --device cuda:i`) song song. `timeout` dừng sạch trước hạn session; chạy lại cell này (hoặc session mới với `RESTORE_FROM`) để tiếp tục.

Ước lượng: T4 chậm hơn RTX 3090 (~34h cho `study` một GPU) → `study` cần nhiều session; thử `quick` trước.

In [ ]:
Path('logs').mkdir(exist_ok=True)
t0 = time.time()
procs = []
for i in range(N_GPU):
    cmd = ['timeout', f'{int(TIME_BUDGET_H*3600)}', sys.executable, '-u', '-m', 'experiments.plan', 'run', SUITE,
           '--shard', f'{i}/{N_GPU}', '--device', f'cuda:{i}', '--num-workers', str(NUM_WORKERS)]
    procs.append((i, subprocess.Popen(cmd, stdout=open(f'logs/shard{i}.log','w'), stderr=subprocess.STDOUT)))
    print('started', ' '.join(cmd))
for i, p in procs: p.wait()
print(f'xong sau {(time.time()-t0)/3600:.2f} h; exit codes', [p.returncode for _, p in procs])

In [ ]:
for i in range(N_GPU):
    print(f'--- shard{i} ---'); print(subprocess.run(['tail','-n','15',f'logs/shard{i}.log'],capture_output=True,text=True).stdout)
print(subprocess.run([sys.executable,'-m','experiments.plan','status',SUITE],capture_output=True,text=True).stdout)

## Tổng hợp bảng/hình (chỉ khi pha đã đủ run)

In [ ]:
left = subprocess.run([sys.executable,'-m','experiments.plan','status',SUITE],capture_output=True,text=True).stdout
done_all = len(set(left.split()[1].split('/')[:1])) and left.split()[1].split('/')[0] == left.split()[1].split('/')[1]
if not done_all:
    print('Suite chưa đủ run -> bỏ qua aggregate; Save Version rồi chạy lại cùng PHASE với RESTORE_FROM.')
else:
    for cmd in PHASES[PHASE]['post']:
        r = subprocess.run([sys.executable,*cmd.split()], capture_output=True, text=True)
        print('$', cmd); print(r.stdout[-3000:], r.stderr[-1500:])

## Đóng gói output
Không nhét checkpoint `.pt` lớn vào zip; JSON kết quả + log + bảng/hình là đủ. Để chạy tiếp session sau, lưu output notebook (Save Version) rồi gắn làm Input và đặt `RESTORE_FROM`.

In [ ]:
bundle = Path('/kaggle/working/degst-results.zip')
with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED,allowZip64=True) as z:
    for base in ('results','logs','paper/tables','paper/Figures'):
        for p in Path(base).rglob('*'):
            if p.is_file() and p.suffix != '.pt': z.write(p, p)
print(bundle, f'{bundle.stat().st_size/2**20:.1f} MiB')